# GDB loader statistics

This notebook explores the mode-based `GDBLoader` interface and draws graph-level statistics for one selected official GDB subset. Unlike ZINC and QM9, GDB is treated as a family of curated downloadable modes rather than one CSV dataset.


In [1]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not find notebooks/_bootstrap.py from the current working tree")

runpy.run_path(str(_bootstrap_path))


{'__name__': '<run_path>',
 '__doc__': 'Shared bootstrap for notebooks in the AbstractGraph ecosystem.',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/home/fabrizio/code/abstractgraph-ecosystem/repos/abstractgraph-graphicalizer/notebooks/_bootstrap.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from abstractgraph_graphicalizer.chem import GDBLoader


## Mode overview

`GDBLoader` exposes a fixed set of supported official modes. The safe default is `lead_like`; `50M` is the recommended larger-scale option when you explicitly want more data.


In [3]:
loader = GDBLoader(on_error="skip")
mode_table = pd.DataFrame(
    {
        "mode": summary.mode,
        "family": summary.family,
        "approx_molecules": summary.approx_molecules,
        "approx_compressed_size": summary.approx_compressed_size,
        "description": summary.description,
        "downloaded": summary.downloaded,
    }
    for summary in loader.list_modes()
).sort_values(["approx_molecules", "mode"], ascending=[False, True]).reset_index(drop=True)

print("root:", loader.root)
display(mode_table)
print(loader.format_mode_table())


root: /home/fabrizio/code/abstractgraph-ecosystem/repos/abstractgraph-graphicalizer/data/GDB


,mode,family,approx_molecules,approx_compressed_size,description,downloaded
0,gdb13_full,GDB-13,977468314,2.6 GB,Entire GDB-13 archive. Advanced multi-GB downl...,False
1,50M,GDB-17,50000000,484 MB,Practical large-scale 50 million molecule GDB-...,False
2,lead_like,GDB-17,11000000,75 MB,Lead-like GDB-17 subset (100-350 MW and 1-3 cl...,False
3,1M,GDB-13,1000000,14.8 MB,Annotated 1 million molecule GDB-13 random sam...,False
4,lead_like_no_small_rings,GDB-17,800000,55 MB,Lead-like GDB-17 subset without small rings (3...,False


mode                         family        molecules       size  downloaded
------------------------------------------------------------------------------
lead_like                    GDB-17       11,000,000      75 MB          no
50M                          GDB-17       50,000,000     484 MB          no
lead_like_no_small_rings     GDB-17          800,000      55 MB          no
1M                           GDB-13        1,000,000    14.8 MB          no
gdb13_full                   GDB-13      977,468,314     2.6 GB          no


## Load one mode and inspect graph statistics

By default this notebook uses the safe `lead_like` mode. If the archive is not present locally, `GDBLoader.load(...)` will download the selected official subset, build the cached graph corpus in streaming mode, and then load the requested slice.


In [ ]:
mode = "lead_like"
limit = 50000
max_node_count = None
chunk_size = 2000

print(loader.describe_mode(mode))

graphs, metadata = loader.load(
    mode,
    limit=limit,
    max_node_count=max_node_count,
    chunk_size=chunk_size,
)

paths = loader.resolve_paths(mode, download=False)

print("mode:", mode)
print("archive_path:", paths.archive_path)
print("graphs loaded:", len(graphs))
display(metadata.head())


lead_like: Lead-like GDB-17 subset (100-350 MW and 1-3 clogP). Recommended safe default. (family=GDB-17, molecules~11,000,000, size~75 MB)


In [ ]:
graph_stats = pd.DataFrame(
    {
        "node_count": [graph.number_of_nodes() for graph in graphs],
        "edge_count": [graph.number_of_edges() for graph in graphs],
    }
)
graph_stats["average_degree"] = 2.0 * graph_stats["edge_count"] / graph_stats["node_count"].clip(lower=1)
graph_stats["density"] = 2.0 * graph_stats["edge_count"] / (
    graph_stats["node_count"] * (graph_stats["node_count"] - 1)
).where(graph_stats["node_count"] > 1, 1)

display(graph_stats.describe().T)


In [ ]:
node_counts = graph_stats["node_count"].astype(int)
edge_counts = graph_stats["edge_count"].astype(int)
node_tick_values = np.arange(node_counts.min(), node_counts.max() + 1, 1)
node_bins = np.arange(node_counts.min() - 0.5, node_counts.max() + 1.5, 1)
edge_tick_values = np.arange(edge_counts.min(), edge_counts.max() + 1, 1)
edge_bins = np.arange(edge_counts.min() - 0.5, edge_counts.max() + 1.5, 1)

fig, axes = plt.subplots(2, 2, figsize=(20, 10))

axes[0, 0].hist(node_counts, bins=node_bins, color="#1f77b4", edgecolor="white")
axes[0, 0].set_title("Node count distribution")
axes[0, 0].set_xlabel("nodes")
axes[0, 0].set_ylabel("graphs")
axes[0, 0].set_xticks(node_tick_values)
axes[0, 0].set_yscale("log")

axes[0, 1].hist(edge_counts, bins=edge_bins, color="#ff7f0e", edgecolor="white")
axes[0, 1].set_title("Edge count distribution")
axes[0, 1].set_xlabel("edges")
axes[0, 1].set_ylabel("graphs")
axes[0, 1].set_xticks(edge_tick_values)
axes[0, 1].set_yscale("log")

axes[1, 0].hist(graph_stats["average_degree"], bins=40, color="#2ca02c", edgecolor="white")
axes[1, 0].set_title("Average degree distribution")
axes[1, 0].set_xlabel("average degree")
axes[1, 0].set_ylabel("graphs")

axes[1, 1].scatter(graph_stats["node_count"], graph_stats["edge_count"], s=8, alpha=0.25, color="#d62728")
axes[1, 1].set_title("Edges vs nodes")
axes[1, 1].set_xlabel("nodes")
axes[1, 1].set_ylabel("edges")

plt.tight_layout()


In [ ]:
summary = pd.DataFrame(
    {
        "graphs": [len(graph_stats)],
        "mean_nodes": [graph_stats["node_count"].mean()],
        "median_nodes": [graph_stats["node_count"].median()],
        "mean_edges": [graph_stats["edge_count"].mean()],
        "median_edges": [graph_stats["edge_count"].median()],
        "mean_density": [graph_stats["density"].mean()],
    },
    index=[mode],
).round(3)

summary
